# VC3 · Demo en directo — Smart City
## "Una alerta nos costó dinero. Antes de fiarnos de la siguiente, vamos a auditarla."

**Esto NO es la actividad evaluable.** Es la demostración de la videoconferencia del NF3.
Tu actividad va de auditar **reportes anti-cheat de un juego** y usa otros datos. Aquí puedes
ejecutar, romper y experimentar: **no se entrega**.

**Antes de ejecutar nada**, una sola vez, desde la raíz del repositorio:

```bash
python demo/nf3_smartcity/preparar_demo.py
```

---

### El encargo

Anoche, el sistema de la ciudad disparó una **alerta roja** de calidad del aire en un distrito.
Se activó el protocolo: se restringió el tráfico, se avisó a los centros de salud. Coste real,
molestias reales.

Esta mañana, la pregunta del jefe de servicio es incómoda: **"¿la alerta era de verdad?"**. Y
la respuesta honesta es: *no lo sé, no lo comprobamos*. En el NF1 y el NF2 hemos calculado un
montón de números. Nunca nos hemos preguntado si eran **de fiar**.

Eso acaba hoy. Vamos a auditar la fiabilidad en tres frentes:
1. **¿Llegaron los datos enteros?** (integridad de los ficheros)
2. **¿El umbral que disparó la alarma era honesto?** (la línea base)
3. **¿La tabla de alertas cumple lo que promete?** (calidad como código)

In [ ]:
import os, json, hashlib, subprocess
from pathlib import Path
import numpy as np
import pandas as pd

try:
    BASE = Path(__file__).parent
except NameError:
    BASE = Path.cwd()
    if BASE.name != "nf3_smartcity":
        BASE = BASE / "demo" / "nf3_smartcity"
RAW = BASE / "raw"

assert (RAW / "manifiesto.json").exists(), (
    "No encuentro los datos. Ejecuta una vez, desde la raíz del repo:\n"
    "    python demo/nf3_smartcity/preparar_demo.py"
)
print("Datos listos en:", BASE)

---
# FRENTE 1 · ¿Llegaron los datos enteros?

El histórico que alimentó la alerta llegó de un sistema distribuido, en trozos. Antes de
calcular **nada**, dos preguntas que casi nadie se hace: ¿están **todos** los trozos? ¿está
cada trozo **intacto**?

## 1a · El marcador `_SUCCESS`

In [ ]:
dist_ok = RAW / "dist_ok"
dist_incompleto = RAW / "dist_incompleto"

print("dist_ok/         :", sorted(p.name for p in dist_ok.iterdir()))
print("dist_incompleto/ :", sorted(p.name for p in dist_incompleto.iterdir()))

Fíjate en la diferencia: `dist_ok/` tiene un fichero vacío llamado **`_SUCCESS`**. El otro no.

Ese fichero no contiene datos. Es un **sello**: lo escribe el sistema distribuido **al terminar
el trabajo entero**, como última acción. Su presencia dice *"el job completó; el directorio
está entero"*.

Y aquí está la pregunta de examen, que tiene trampa: **¿qué se infiere de su ausencia?**

In [ ]:
def tiene_success(directorio):
    return (directorio / "_SUCCESS").exists()

print("¿dist_ok completo?        ", tiene_success(dist_ok))
print("¿dist_incompleto completo?", tiene_success(dist_incompleto))
print()
print("dist_incompleto tiene", len(list(dist_incompleto.glob('part-*.csv'))), "ficheros DENTRO...")
print("...pero sin _SUCCESS, NO sabemos si son 2 de 2, o 2 de 500.")

Lee esto despacio, porque es donde cae la mayoría:

> La ausencia de `_SUCCESS` **no significa "no hay datos"**. Hay dos ficheros ahí dentro,
> perfectamente legibles. Significa que **nadie ha declarado el directorio como completo**. Los
> datos que hay pueden ser una **fracción arbitraria** de los que debería haber —el job murió a
> la mitad, o se está escribiendo ahora mismo—. Usarlos es calcular una media sobre un trozo al
> azar y creértela.

**Un directorio sin `_SUCCESS` no es un directorio con menos datos. Es un directorio en el que
no puedes confiar.**

## 1b · Los checksums: ¿está cada fichero intacto?

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for bloque in iter(lambda: f.read(8192), b""):
            h.update(bloque)
    return h.hexdigest()

# El manifiesto se firmó cuando los datos eran correctos. Lo comparamos con AHORA.
manifiesto = json.load(open(RAW / "manifiesto.json"))

print(f"{'fichero':<14} {'esperado':<12} {'actual':<12} veredicto")
corruptos = []
for nombre, hash_esperado in manifiesto.items():
    hash_actual = sha256(dist_ok / nombre)
    ok = hash_actual == hash_esperado
    if not ok:
        corruptos.append(nombre)
    print(f"{nombre:<14} {hash_esperado[:10]:<12} {hash_actual[:10]:<12} {'OK' if ok else '<-- CORRUPTO'}")

print(f"\nFicheros corruptos: {corruptos}")

**`part-2.csv` está corrupto.** El `_SUCCESS` estaba —el job terminó— y aun así un fichero se
dañó *después*: una transferencia a medias, un bit que se volteó en el disco. El marcador dice
"completo"; el checksum dice "pero uno está roto".

> **Los dos controles responden preguntas distintas, y necesitas los dos.** `_SUCCESS`:
> *¿están todos?* Checksum: *¿está cada uno intacto?* Un directorio puede tener `_SUCCESS` y
> ficheros corruptos (lo acabas de ver). Y puede tener todos los checksums correctos pero
> faltarle la mitad de los ficheros (si no hay `_SUCCESS`, no lo sabes).

**Veredicto del frente 1: estos datos NO son fiables.** Y, sin embargo, con ellos se disparó
una alerta que costó dinero.

---
# FRENTE 2 · ¿El umbral que disparó la alarma era honesto?

La alerta saltó porque un sensor superó un **umbral**. ¿De dónde salió ese umbral? Del método
clásico: **media ± 3σ** del histórico del sensor. Suena estadísticamente impecable. Vamos a ver
por qué es una trampa, con un sensor de **velocidad de tráfico** que sabemos que tuvo atascos.

In [ ]:
x = pd.read_csv(RAW / "sensor_velocidad.csv")["velocidad"].values
print(f"{len(x)} lecturas. Sabemos que hubo 40 atascos reales (velocidad ~8 km/h).")
print(f"lecturas normales rondan los 45 km/h\n")

# Método 1: media ± 3σ. Un atasco es velocidad ANORMALMENTE BAJA -> miramos el umbral inferior.
media, sigma = x.mean(), x.std()
umbral_media = media - 3 * sigma
anom_media = int((x < umbral_media).sum())
print(f"MÉTODO 1 · media ± 3σ")
print(f"  media = {media:.1f}   σ = {sigma:.1f}")
print(f"  umbral inferior = {umbral_media:.1f}")
print(f"  atascos detectados: {anom_media}   <-- ¿¿de 40??")

**Cero.** El método que suena más riguroso **no ha detectado ni un solo atasco**, teniendo 40
delante. ¿Cómo es posible?

Porque los 40 atascos, al estar tan lejos de la normalidad, **inflan la propia σ** que usamos
para poner el umbral. Con una σ enorme, `media − 3σ` se va a un número negativo: un umbral que
**ninguna velocidad puede superar por abajo**. Los atascos han **saboteado el detector que
debía cazarlos**.

> Esto es el **enmascaramiento estadístico**: los valores anómalos, al participar en el cálculo
> de la línea base, la corrompen — y la línea base corrompida ya no puede detectarlos. **La
> anomalía se esconde a sí misma.** Y lo perverso: cuantos más atascos hay, menos detecta. Un
> sistema de alertas que falla más cuanto peor van las cosas.

In [ ]:
# Método 2: mediana y MAD (estadísticos ROBUSTOS)
mediana = np.median(x)
mad = np.median(np.abs(x - mediana))       # mediana de las distancias a la mediana
umbral_robusto = mediana - 3 * (1.4826 * mad)
anom_robusto = int((x < umbral_robusto).sum())

print(f"MÉTODO 2 · mediana + 3·(1.4826·MAD)")
print(f"  mediana = {mediana:.1f}   MAD = {mad:.2f}")
print(f"  umbral inferior = {umbral_robusto:.1f}")
print(f"  atascos detectados: {anom_robusto}   <-- los 40")

**Los 40.** El mismo dato, la misma regla de "3 desviaciones", pero con una **vara de medir que
no se deja engañar**.

Dos claves para el examen:

- **El `1.4826` no es magia.** Es una conversión de unidades: en una distribución normal,
  σ ≈ 1,4826 · MAD. Multiplicar por ese factor pone el MAD "en unidades de σ" para que tu regla
  de las 3 desviaciones siga significando lo mismo. No cambias de regla; **cambias la vara**.
- **El término que lo resume todo: punto de ruptura.** La media tiene punto de ruptura del
  **0 %**: *un solo* valor infinito la manda al infinito. La mediana aguanta hasta el **50 %**
  de contaminación sin moverse. Por eso, ante outliers, mediana y no media. Esa frase —"la media
  tiene punto de ruptura 0 %"— es lo que separa un aprobado de un excelente.

**Veredicto del frente 2: el umbral que disparó la alarma probablemente era basura.** O bien
saltó por ruido, o bien —peor— llevaba días sin detectar atascos reales.

---
# FRENTE 3 · ¿La tabla de alertas cumple lo que promete?

Los dos frentes anteriores los hemos comprobado **a mano**, con código. Funciona una vez. Pero
un control de calidad que se ejecuta "cuando alguien se acuerda" **no es un control, es un buen
propósito**. El frente 3 es hacer que la calidad **se compruebe sola, cada vez, versionada en
git**. Esa es la herramienta del núcleo: **dbt**.

La idea, entera, en una frase:

> **Un test de dbt es una consulta SQL que devuelve las filas que están MAL. Si devuelve cero
> filas, el test pasa.**

Tenemos un proyecto dbt ya montado en `demo/nf3_smartcity/dbt_calidad/`. Vamos a mirarlo y a
ejecutarlo.

In [ ]:
DBT = BASE / "dbt_calidad"

print("=== models/schema.yml (tests GENÉRICOS, declarados en YAML) ===\n")
print((DBT / "models" / "schema.yml").read_text())

Lee ese YAML como leería el jefe de servicio, que no programa: **dice qué promete la tabla**.
`id_alerta` es único y no nulo. `severidad` no es nula y solo puede ser una de cuatro. No hay
que leer código para saberlo — y eso, no el ahorro de líneas, es lo que hace *declarativo* el
enfoque. Cada test cubre una **dimensión de calidad**: `unique`→unicidad, `not_null`→
completitud, `accepted_values`→validez.

In [ ]:
print("=== tests/aqi_en_rango.sql (test SINGULAR, SQL a medida) ===\n")
print((DBT / "tests" / "aqi_en_rango.sql").read_text())

Ese es un **test singular**: para una regla de negocio que ningún test de serie cubre (el índice
de calidad del aire debe estar entre 0 y 500). Es SQL que selecciona **lo que no debería
existir**. Cero filas = pasa.

Y ahora lo que ningún documento puede enseñar: **ejecutarlo y ver el pipeline ponerse en rojo.**

In [ ]:
# dbt corre sobre DuckDB: sin servidor, sin configurar nada, un fichero local.
import re
DBT = (BASE / "dbt_calidad").resolve()

assert (DBT / "dbt_project.yml").exists(), (
    f"No encuentro dbt_project.yml en {DBT}.\n"
    "Comprueba que la carpeta dbt_calidad/ se subió COMPLETA al repo:\n"
    "    dbt_project.yml, profiles.yml, models/ (schema.yml + stg_alertas.sql), tests/"
)

def dbt(cmd):
    """Ejecuta dbt con rutas ABSOLUTAS (no depende del directorio de trabajo)."""
    r = subprocess.run(
        ["dbt"] + cmd.split(),
        cwd=str(DBT),
        env={**os.environ, "DBT_PROFILES_DIR": str(DBT)},   # <- absoluto, no "."
        capture_output=True, text=True,
    )
    return r.stdout + r.stderr

def confirma(salida, paso):
    """dbt con --quiet no imprime nada si va bien; confirmamos el paso a mano."""
    ok = "Completed successfully" in salida
    if ok:
        print(f"[OK] {paso}")
    else:
        print(f"[!] {paso} — revisa la salida:")
        print(re.sub(r"\x1b\[[0-9;]*m", "", salida)[-500:])
    return ok

confirma(dbt("seed"), "Datos de ejemplo cargados (dbt seed)")
confirma(dbt("run"),  "Modelo construido (dbt run)")

In [ ]:
salida = dbt("test")
# Nos quedamos con las líneas de resultado de cada test
for linea in salida.splitlines():
    if any(k in linea for k in ("PASS", "FAIL", "Done")):
        # limpiamos códigos de color
        import re
        print(re.sub(r"\x1b\[[0-9;]*m", "", linea).split("  ", 1)[-1])

**Ahí está el control.** No un informe: un **control**. `dbt test` ha devuelto:

- `not_null_severidad` → **60 filas** malas (60 alertas sin severidad).
- `unique_id_alerta` → **30 filas** (30 identificadores duplicados).
- `aqi_en_rango` → **176 filas** con el índice fuera de [0, 500].
- `accepted_values_severidad` → **1** categoría inventada.
- Solo pasa `not_null_id_alerta`.

Y lo más importante, invisible en la salida pero decisivo: cuatro tests están en **`severity:
error`** (el valor por defecto). Eso significa que, en un pipeline real, **estos datos no se
publican**. El pipeline se **para** aquí, antes de que ninguna alerta salga a la calle.

> **La decisión de ingeniería es la severidad.** `error` para lo que rompe el contrato (se
> para el mundo); `warn` para lo que conviene vigilar (se registra y sigue). Reservar `error`
> solo para lo grave es lo que evita la **fatiga de alertas**: si todo es crítico, nada lo es.
> Lo veremos a fondo en el NF4.

---
## Lo que ha pasado en esta hora

Una alerta que costó dinero. Y tres preguntas que nadie había hecho:

| Frente | La pregunta | La herramienta | Teoría |
|---|---|---|---|
| 1 | ¿Llegaron los datos enteros? | `_SUCCESS` + checksums | §3.4 |
| 2 | ¿El umbral era honesto? | mediana + MAD (robustos) | §3.2 |
| 3 | ¿La tabla cumple su promesa? | tests de dbt, `severity: error` | §3.6 |

La idea que une las tres, y con la que cierro: **la calidad no se arregla en el dashboard, se
asegura en el pipeline.** Un dashboard precioso sobre datos que no pasaron estos tres frentes es
una mentira con colores — y esa mentira se descubre en la reunión, no en el código.

**Y ahora te toca a ti**, auditando los reportes anti-cheat de un juego. Mismo trabajo, otros
datos: `_SUCCESS`, checksums, baseline robusto y tus propios tests de dbt.

> **El gancho del NF4:** todos estos controles se ejecutan **una vez**, cuando tú los lanzas.
> ¿Y quién vigila el sistema **el resto del tiempo**, a las tres de la madrugada, cuando nadie
> mira? Eso es monitorización, y es el próximo núcleo.